<a href="https://colab.research.google.com/github/chandrakant131103/dental-ai-diagnostics/blob/main/notebooks/01_full_pipeline_colab_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dental Diagnostic AI - End-to-End Colab Pipeline

Runs the full pipeline: Kaggle download -> preprocessing -> augmentation -> YOLO detection training -> U-Net segmentation training -> evaluation -> Grad-CAM demo -> deploy API + Streamlit app via ngrok.

**Set Runtime > Change runtime type > GPU (T4) before running.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
# 1. Remount Drive (if not already done this session)
from google.colab import drive
drive.mount('/content/drive')

# 2. Re-clone the repo
!git clone https://github.com/chandrakant131103/dental-ai-diagnostics.git
%cd /content/dental-ai-diagnostics
!pip install -q -r requirements.txt

# 3. Confirm data/ actually exists now
!ls data/

Mounted at /content/drive
Cloning into 'dental-ai-diagnostics'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 91 (delta 37), reused 36 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 71.97 KiB | 3.43 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/content/dental-ai-diagnostics
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.2/280.2 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 0. Clone repo and install dependencies

In [ ]:
!git clone https://github.com/chandrakant131103/dental-ai-diagnostics.git
%cd dental-ai-diagnostics
!pip install -q -r requirements.txt

Cloning into 'dental-ai-diagnostics'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 56 (delta 15), reused 32 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 39.34 KiB | 9.83 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/dental-ai-diagnostics
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.2/280.2 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 130.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 10.7 MB

In [ ]:
%cd /content/dental-ai-diagnostics
!git pull
!nproc

/content/dental-ai-diagnostics
Already up to date.
2


## 1. Kaggle setup and dataset download
Upload your `kaggle.json` (from kaggle.com/settings > API > Create New Token) when prompted.

In [2]:
from google.colab import files
files.upload()  # upload kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json


Saving kaggle.json to kaggle.json


In [3]:
!python data/kaggle_download.py --dataset reemsalahshehab/dental-datasetv6 --out /content/data --fix-yaml


[kaggle_download] Running: kaggle datasets download -d reemsalahshehab/dental-datasetv6 -p /content/data --unzip
Dataset URL: https://www.kaggle.com/datasets/reemsalahshehab/dental-datasetv6
License(s): apache-2.0


[kaggle_download] OK. train=4772 valid=2071 test=1580 images. data.yaml found at /content/data/data.yaml
[kaggle_download] Rewrote /content/data/data.yaml


## 2. Preprocessing (CLAHE + denoise + letterbox, labels remapped in lockstep)

In [ ]:
import os
for split in ['train', 'valid', 'test']:
    os.system(
        f'python data/preprocess.py '
        f'--src /content/data/{split}/images --dst /content/data_proc/{split}/images '
        f'--labels-src /content/data/{split}/labels --labels-dst /content/data_proc/{split}/labels '
        f'--size 1024 --progress-every 200'
    )
print('Preprocessing done.')


Preprocessing done.


In [ ]:
!find /content/data_proc/train/images -type f | wc -l
!find /content/data_proc/valid/images -type f | wc -l
!find /content/data_proc/test/images -type f | wc -l

4772
2071
1580


In [ ]:
# 1. Do raw Kaggle labels even have content?
!ls /content/data/train/labels | head -3
!cat /content/data/train/labels/$(ls /content/data/train/labels | head -1)

# 2. Does every image have a matching label filename (stem match)?
!ls /content/data/train/images | head -3
!ls /content/data/train/labels | head -3

# 3. Count how many raw label files are actually non-empty
!find /content/data/train/labels -name "*.txt" -size +0c | wc -l
!find /content/data/train/labels -name "*.txt" | wc -l

# 4. Check the preprocessed labels (before augmentation)
!find /content/data_proc/train/labels -name "*.txt" -size +0c | wc -l
!cat /content/data_proc/train/labels/$(ls /content/data_proc/train/labels | head -1)

# 5. Check the augmented labels (what training actually reads)
!find /content/data_aug/train/labels -name "*.txt" -size +0c | wc -l
!cat /content/data_aug/data.yaml

00cf39c1-Karaptiyan_Robert_50yo_13032021_185908_jpg.rf.98b2e72cb9a26e75d40df97e04473ada.txt
00cf39c1-Karaptiyan_Robert_50yo_13032021_185908_jpg.rf.d93e04580a7164c9e5c7ebcd57177f6a.txt
01b0dd74-Gazavandi_Neda_2022-05-14190158_jpg.rf.93976cb5376c518410a89cbbc608a0a8.txt
2 0.23267968749999998 0.361446875 0.24760624999999997 0.3741 0.2583546875 0.37663125000000003 0.2619375 0.363978125 0.25298125 0.34879375 0.24880156250000002 0.347528125 0.2422328125 0.347528125 0.2308875 0.3412015625 0.23267968749999998 0.361446875
2 0.2679078125 0.43483593750000005 0.270296875 0.41712187500000003 0.24103906249999998 0.386753125 0.23745625 0.4019375 0.2679078125 0.43483593750000005
1 0.3085109375 0.3930796875 0.3108984375 0.36524375 0.3001515625 0.347528125 0.27985000000000004 0.338671875 0.2720875 0.34499843750000003 0.26313125 0.363978125 0.2661171875 0.3842234375 0.2720875 0.3930796875 0.290596875 0.4107953125 0.3007484375 0.4069984375 0.3085109375 0.3930796875
1 0.37896718749999997 0.4133250000000000

## 3. Augmentation (train split only)

In [ ]:
!rm -rf /content/data_aug
%cd /content/dental-ai-diagnostics
!git pull

/content/dental-ai-diagnostics
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 9 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 2.84 KiB | 969.00 KiB/s, done.
From https://github.com/chandrakant131103/dental-ai-diagnostics
   2111740..68de3a5  main       -> origin/main
Updating 2111740..68de3a5
Fast-forward
 data/augment.py           |  5 ++++-
 models/detection/train.py | 34 ++++++++++++++++++++++++----------
 2 files changed, 28 insertions(+), 11 deletions(-)


In [ ]:
!python data/augment.py \
  --images /content/data_proc/train/images --labels /content/data_proc/train/labels \
  --out-images /content/data_aug/train/images --out-labels /content/data_aug/train/labels \
  --multiplier 3


[augment] 4772 source images, x3 augmentations each
 29% 1404/4772 [02:35<07:03,  7.95it/s]/usr/local/lib/python3.12/dist-packages/albumentations/augmentations/dropout/functional.py:559: RuntimeWarning: invalid value encountered in divide
  visibility_ratios = remaining_areas / box_areas
100% 4772/4772 [08:56<00:00,  8.90it/s]
[augment] Done -> /content/data_aug/train/images


In [ ]:
# point augmented train split back at the original data.yaml structure
import shutil, yaml
shutil.copytree('/content/data_proc/valid', '/content/data_aug/valid', dirs_exist_ok=True)
shutil.copytree('/content/data_proc/test', '/content/data_aug/test', dirs_exist_ok=True)
shutil.copy('/content/data/data.yaml', '/content/data_aug/data.yaml')


'/content/data_aug/data.yaml'

## 4. Stage 1 - Train YOLO tooth/finding detector

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q kaggle ultralytics
from google.colab import files
files.upload()  # kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d reemsalahshehab/dental-datasetv6 -p /content/data --unzip

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.3 MB/s eta 0:00:00


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/reemsalahshehab/dental-datasetv6
License(s): apache-2.0
100% 354M/354M [00:06<00:00, 59.5MB/s]



In [ ]:
%cd /content/dental-ai-diagnostics
!ls models/detection/train.py

/content/dental-ai-diagnostics
models/detection/train.py


In [ ]:
!python models/detection/train.py \
  --data /content/data_aug/data.yaml --model yolov8s.pt \
  --epochs 30 --imgsz 640 --batch 16 \
  --project /content/drive/MyDrive/dental-ai-diagnostics/runs/detect

[train] Class distribution (train split):
   0 Caries               15952
   1 Crown                18464
   2 Filling              61432
   3 Implant              4072
   4 Malaligned           8
   5 Mandibular Canal     2024
   6 Missing teeth        7327
   7 Periapical lesion    10280
   8 Retained root        8
   9 Root Canal Treatment 30201
  10 Root Piece           5864
  12 impacted tooth       35672
  13 maxillary sinus      1536
[train] Max/min class imbalance ratio: 7679.0x
[train] Severe imbalance detected -> recommend focal loss / oversampling rare classes
Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data_aug/data.yaml

In [ ]:
!ls -la /content/drive/MyDrive/dental-ai-diagnostics/runs/detect/tooth_localization/weights/
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/dental-ai-diagnostics/runs/detect/tooth_localization/weights/last.pt")
results = model.train(resume=True)

total 43995
-rw------- 1 root root 22525226 Jul 11 20:44 best.pt
-rw------- 1 root root 22525226 Jul 11 20:44 last.pt
WARNING ⚠️ model '/content/drive/MyDrive/dental-ai-diagnostics/runs/detect/tooth_localization/weights/last.pt' is not a resumable training checkpoint (missing epoch/optimizer state). Use 'resume' only to continue incomplete training. Starting new training instead.
Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, 

In [ ]:
%%writefile /content/dental-ai-diagnostics/data/yolo_to_coco.py
"""
Convert YOLO-format labels (boxes OR polygons, normalized 0-1) into a COCO-style
JSON with per-instance `segmentation` fields, so Stage 5 (U-Net) can train
against real masks derived from your existing dental-datasetv6 labels.
"""
import argparse
import json
from pathlib import Path

import cv2
import yaml
from tqdm import tqdm


def load_class_names(data_yaml_path: str):
    cfg = yaml.safe_load(Path(data_yaml_path).read_text())
    names = cfg["names"]
    if isinstance(names, dict):
        return {int(k): v for k, v in names.items()}
    return {i: n for i, n in enumerate(names)}


def polygon_area(xy_pairs):
    n = len(xy_pairs) // 2
    if n < 3:
        return 0.0
    area = 0.0
    for i in range(n):
        x1, y1 = xy_pairs[2 * i], xy_pairs[2 * i + 1]
        x2, y2 = xy_pairs[2 * ((i + 1) % n)], xy_pairs[2 * ((i + 1) % n) + 1]
        area += x1 * y2 - x2 * y1
    return abs(area) / 2.0


def yolo_line_to_coco_ann(values, img_w, img_h):
    if len(values) == 4:
        cx, cy, w, h = values
        cx, cy, w, h = cx * img_w, cy * img_h, w * img_w, h * img_h
        x0, y0 = cx - w / 2, cy - h / 2
        poly = [x0, y0, x0 + w, y0, x0 + w, y0 + h, x0, y0 + h]
        bbox = [x0, y0, w, h]
        area = w * h
        return [poly], bbox, area

    if len(values) % 2 != 0 or len(values) < 6:
        return None

    xs = values[0::2]
    ys = values[1::2]
    xs_px = [x * img_w for x in xs]
    ys_px = [y * img_h for y in ys]
    poly = []
    for x, y in zip(xs_px, ys_px):
        poly.extend([x, y])

    x0, x1 = min(xs_px), max(xs_px)
    y0, y1 = min(ys_px), max(ys_px)
    bbox = [x0, y0, x1 - x0, y1 - y0]
    area = polygon_area(poly)
    return [poly], bbox, area


def main(args):
    images_dir = Path(args.images)
    labels_dir = Path(args.labels)
    class_names = load_class_names(args.data_yaml)

    categories = [{"id": cid, "name": name} for cid, name in sorted(class_names.items())]

    images = []
    annotations = []
    ann_id = 1
    img_id = 1

    label_files = sorted(labels_dir.glob("*.txt"))
    skipped_no_image = 0
    skipped_bad_lines = 0

    for label_path in tqdm(label_files, desc=f"Converting {labels_dir.name}"):
        stem = label_path.stem
        img_path = None
        for ext in (".jpg", ".jpeg", ".png"):
            candidate = images_dir / f"{stem}{ext}"
            if candidate.exists():
                img_path = candidate
                break
        if img_path is None:
            skipped_no_image += 1
            continue

        img = cv2.imread(str(img_path))
        if img is None:
            skipped_no_image += 1
            continue
        h, w = img.shape[:2]

        images.append({
            "id": img_id,
            "file_name": img_path.name,
            "width": w,
            "height": h,
        })

        for line in label_path.read_text().splitlines():
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            try:
                cls_id = int(round(float(parts[0])))
                values = [float(v) for v in parts[1:]]
            except ValueError:
                skipped_bad_lines += 1
                continue

            result = yolo_line_to_coco_ann(values, w, h)
            if result is None:
                skipped_bad_lines += 1
                continue
            segmentation, bbox, area = result

            annotations.append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": cls_id,
                "segmentation": segmentation,
                "bbox": bbox,
                "area": area,
                "iscrowd": 0,
            })
            ann_id += 1

        img_id += 1

    coco = {
        "images": images,
        "annotations": annotations,
        "categories": categories,
    }

    out_path = Path(args.out)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(coco))

    print(f"[yolo_to_coco] {len(images)} images, {len(annotations)} annotations -> {out_path}")
    if skipped_no_image:
        print(f"[yolo_to_coco] WARNING: {skipped_no_image} label files had no matching image, skipped")
    if skipped_bad_lines:
        print(f"[yolo_to_coco] WARNING: {skipped_bad_lines} malformed label lines skipped")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--images", required=True)
    parser.add_argument("--labels", required=True)
    parser.add_argument("--data-yaml", required=True)
    parser.add_argument("--out", required=True)
    args = parser.parse_args()
    main(args)

Writing /content/dental-ai-diagnostics/data/yolo_to_coco.py


## 5. Stage 2 - Train U-Net pathology segmentation (uses COCO masks)

In [5]:
!pip install -q pycocotools

!python data/yolo_to_coco.py \
  --images /content/data/train/images --labels /content/data/train/labels \
  --data-yaml /content/data/data.yaml --out /content/data/train/train_coco.json

!python data/yolo_to_coco.py \
  --images /content/data/valid/images --labels /content/data/valid/labels \
  --data-yaml /content/data/data.yaml --out /content/data/valid/valid_coco.json

%cd models/segmentation
!python train.py \
  --images /content/data/train/images --coco /content/data/train/train_coco.json \
  --val-images /content/data/valid/images --val-coco /content/data/valid/valid_coco.json \
  --epochs 40 --batch 8 \
  --out /content/drive/MyDrive/dental-ai-diagnostics/runs/segment
%cd /content/dental-ai-diagnostics

Converting labels: 100% 4772/4772 [00:11<00:00, 412.87it/s]
[yolo_to_coco] 4772 images, 48212 annotations -> /content/data/train/train_coco.json
Converting labels: 100% 2071/2071 [00:04<00:00, 490.17it/s]
[yolo_to_coco] 2071 images, 19300 annotations -> /content/data/valid/valid_coco.json
/content/dental-ai-diagnostics/models/segmentation
[seg-train] device=cuda
loading annotations into memory...
Done (t=1.00s)
creating index...
index created!
loading annotations into memory...
Done (t=0.41s)
creating index...
index created!
[seg-train] 4772 train / 2071 val images, 15 classes (incl. bg)
[seg-train] epoch 1/40 train_loss=2.0134 val_loss=1.3337
[seg-train] saved new best -> /content/drive/MyDrive/dental-ai-diagnostics/runs/segment/unet_best.pt
[seg-train] epoch 2/40 train_loss=1.1362 val_loss=1.0023
[seg-train] saved new best -> /content/drive/MyDrive/dental-ai-diagnostics/runs/segment/unet_best.pt
[seg-train] epoch 3/40 train_loss=0.9461 val_loss=0.9048
[seg-train] saved new best -> /c

## 6. Evaluation (fixed IoU matching, per-class metrics, confusion matrix)

In [7]:
!ls /content/drive/MyDrive/dental-ai-diagnostics/runs/detect/tooth_localization/weights/

best.pt  last.pt


In [8]:
import sys; sys.path.append('.')
import supervision as sv
from ultralytics import YOLO
from evaluation.metrics import evaluate_detections, compute_map
import yaml

data_cfg = yaml.safe_load(open('/content/data/data.yaml'))
class_names = data_cfg['names'] if isinstance(data_cfg['names'], dict) else dict(enumerate(data_cfg['names']))

model = YOLO('/content/drive/MyDrive/dental-ai-diagnostics/runs/detect/tooth_localization/weights/best.pt')
test_ds = sv.DetectionDataset.from_yolo(
    images_directory_path='/content/data/test/images',
    annotations_directory_path='/content/data/test/labels',
    data_yaml_path='/content/data/data.yaml',
)

gts, preds = [], []
for path, image, annotation in test_ds:
    result = model(image, conf=0.25, verbose=False)[0]
    preds.append(sv.Detections.from_ultralytics(result))
    gts.append(annotation)

report = evaluate_detections(gts, preds, num_classes=len(class_names), class_names=class_names)
import pandas as pd
pd.DataFrame(report['per_class'])

,class,tp,fp,fn,precision,recall,f1
0,Caries,189,99,1005,0.6562,0.1583,0.2551
1,Crown,1031,565,126,0.6460,0.8911,0.7490
2,Filling,2463,2394,3197,0.5071,0.4352,0.4684
3,Implant,160,49,18,0.7656,0.8989,0.8269
4,Malaligned,0,0,3,0.0000,0.0000,0.0000
5,Mandibular Canal,26,77,14,0.2524,0.6500,0.3636
6,Missing teeth,182,86,156,0.6791,0.5385,0.6007
7,Periapical lesion,55,53,508,0.5093,0.0977,0.1639
8,Retained root,0,0,35,0.0000,0.0000,0.0000
9,Root Canal Treatment,1288,1334,943,0.4912,0.5773,0.5308


## 7. Grad-CAM explainability demo

In [ ]:
import torch, cv2, matplotlib.pyplot as plt
from explainability.gradcam import gradcam_unet, overlay_heatmap
from models.segmentation.unet import UNet

ckpt = torch.load('models/segmentation/runs/segment/unet_best.pt', map_location='cpu')
unet = UNet(n_channels=1, n_classes=ckpt['num_classes'])
unet.load_state_dict(ckpt['model_state'])
unet.eval()

sample_path = test_ds.images[list(test_ds.images.keys())[0]]  # replace with any crop path
# ... load a tooth crop, run gradcam_unet(unet, tensor, target_class=1), overlay, plt.imshow(...)


## 8. Deploy API + Streamlit app (ngrok tunnel for Colab demo)

In [ ]:
!pip install -q pyngrok
from pyngrok import ngrok
import os

# Set your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token('YOUR_NGROK_TOKEN')

os.environ['DETECTOR_WEIGHTS'] = 'runs/detect/tooth_localization/weights/best.pt'
os.environ['SEGMENTER_WEIGHTS'] = 'models/segmentation/runs/segment/unet_best.pt'

get_ipython().system_raw('uvicorn api.main:app --host 0.0.0.0 --port 8000 &')
api_tunnel = ngrok.connect(8000)
print('API:', api_tunnel.public_url)


In [ ]:
os.environ['DENTAL_API_URL'] = api_tunnel.public_url
get_ipython().system_raw('streamlit run app/streamlit_app.py --server.port 8501 &')
app_tunnel = ngrok.connect(8501)
print('App:', app_tunnel.public_url)


## 9. (Optional) Push trained weights + metrics to your GitHub repo for the README

In [ ]:
# copy best.pt files, run_metadata.json, and the metrics report into results/
# then git add/commit/push from this Colab session or download and push locally
